<b>Quiz 3 - Problem 2</b>

In [1]:
# data

import csv
f = open("quiz 3 problem 2.csv")
csvfile = csv.DictReader(f, delimiter=',')
headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()

# Create set of Nodes.
Nodes = set()
# Create dictionary "Distance" for each arc.
# We can borrow the keys from this dictionary to define the arc set of the network.
Distance = {}
for row in table:
    fromNode = row['From']
    toNode = row['To']
    Distance[(fromNode,toNode)] = float(row['Distance (miles)'])
    Nodes.add(fromNode)
    Nodes.add(toNode)

Arcs = Distance.keys()

In [2]:
Nodes

{'1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9', 'P'}

In [3]:
Distance

{('1', '3'): 80.0,
 ('2', '4'): 68.0,
 ('3', '4'): 54.0,
 ('3', '5'): 94.0,
 ('3', '6'): 91.0,
 ('4', '3'): 51.0,
 ('4', '6'): 57.0,
 ('4', '7'): 116.0,
 ('5', '8'): 17.0,
 ('6', '9'): 15.0,
 ('7', '10'): 13.0,
 ('8', '11'): 71.0,
 ('8', '12'): 139.0,
 ('9', '11'): 92.0,
 ('9', '12'): 74.0,
 ('10', '12'): 83.0,
 ('11', 'P'): 62.0,
 ('12', 'P'): 47.0}

In [4]:
Arcs

dict_keys([('1', '3'), ('2', '4'), ('3', '4'), ('3', '5'), ('3', '6'), ('4', '3'), ('4', '6'), ('4', '7'), ('5', '8'), ('6', '9'), ('7', '10'), ('8', '11'), ('8', '12'), ('9', '11'), ('9', '12'), ('10', '12'), ('11', 'P'), ('12', 'P')])

In [5]:
Supply = {'8': 0, '1': -1, '6': 0,'P': 2,'12': 0,'5': 0, '10': 0,'2': -1,'7': 0,'4': 0,'11': 0,'3': 0,'9': 0}

In [13]:
from docplex.mp.model import Model
mdl = Model()

In [14]:
# variables
leg = mdl.continuous_var_dict(Arcs, lb=0, ub=2,  name='leg')

In [15]:
# objective
mdl.minimize(mdl.sum(Distance[i,j]*leg[i,j] for (i,j) in Arcs))

In [16]:
#max leg constraint
mdl.add_constraint(mdl.sum(leg[i,j] for [i,j] in Arcs if (i,j) == ('5', '8')) ==0)
mdl.add_constraint(mdl.sum(leg[i,j] for [i,j] in Arcs if (i,j) == ('6', '9')) <=1)
mdl.add_constraint(mdl.sum(leg[i,j] for [i,j] in Arcs if (i,j) == ('7', '10')) <=1)

docplex.mp.LinearConstraint[](leg_7_10,LE,1)

In [17]:
# flow conservation constraint
for j in Nodes:
    inflow = mdl.sum(leg[i,j] for i in Nodes if (i,j) in Arcs)
    outflow = mdl.sum(leg[j,i] for i in Nodes if (j,i) in Arcs)
    mdl.add_constraint(inflow - outflow == Supply[j])

In [18]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.00258803,status='optimal')

In [12]:
mdl.print_solution()

objective: 585.000
status: OPTIMAL_SOLUTION(2)
  leg_1_3=1.000
  leg_2_4=1.000
  leg_3_5=1.000
  leg_4_6=1.000
  leg_5_8=1.000
  leg_6_9=1.000
  leg_8_11=1.000
  leg_9_12=1.000
  leg_11_P=1.000
  leg_12_P=1.000


In [19]:
mdl.print_solution()

objective: 634.000
status: OPTIMAL_SOLUTION(2)
  leg_1_3=1.000
  leg_2_4=1.000
  leg_3_6=1.000
  leg_4_7=1.000
  leg_6_9=1.000
  leg_7_10=1.000
  leg_9_12=1.000
  leg_10_12=1.000
  leg_12_P=2.000
